# Dia 14 — Trabalhando com o sistema de arquivos

**🃏 Cartas:** 14 · 35 · 56 &nbsp;·&nbsp; **📘 Original:** *Day 14. Working with the file system*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 14 — A sala de evidências

Duas semanas de NFA e a pasta do caso já virou um labirinto: depoimentos, a conversa, a cópia da
vitrine, o dump do painel, a planilha das vítimas, CSVs que você mesmo gerou. A Iara abre o
explorador de arquivos e cruza os braços.

— Se a Delegada Rita perguntar "em quais arquivos aparece a chave `f38b2ffc`?", quanto tempo você
leva para responder?

Você começa a abrir arquivo por arquivo. Ela interrompe no terceiro.

— Agora a pergunta que importa de verdade: como você **prova** que este depoimento é o mesmo que a
vítima entregou, e que ninguém trocou uma letra no meio do caminho?

Ela escreve no quadro: **SHA-256**.

— É a impressão digital do arquivo. Mudou um byte, mudou o hash inteiro. O Código de Processo Penal
chama isso de **cadeia de custódia**: cada vestígio precisa ter a história documentada, de quem
coletou até quem analisou. Para prova digital, o hash é o lacre.

Hoje você vai fazer o Python andar pelas pastas sozinho, achar padrões em tudo e lacrar cada arquivo
com um manifesto. Amanhã, quando alguém duvidar, o manifesto responde.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-14-sistema-de-arquivos")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Rodar os quatro exemplos originais com `glob` e `os`.
- Usar `pathlib`: caminhos com `/`, `glob`, `rglob`, `stat`, `read_text`.
- Varrer uma pasta inteira atrás de CPFs, chaves PIX e e-mails (regex do Dia 11).
- Calcular **SHA-256** em blocos com `hashlib` e gerar um **manifesto de custódia**.
- Verificar depois se algum arquivo foi alterado, sumiu ou apareceu.

## 🧠 Conceito em 2 minutos

**Andar por pastas.** O original usa `os.chdir("test_dir")` + `glob.glob("*.txt")`. Funciona, mas o
`chdir` muda a pasta do programa inteiro. O `pathlib` faz o mesmo sem mudar de pasta:

| Original (`os` + `glob`) | `pathlib` |
|---|---|
| `os.chdir("test_dir")` + `glob.glob("*.txt")` | `Path("test_dir").glob("*.txt")` |
| `glob.glob("**/*.txt", recursive=True)` | `Path("test_dir").rglob("*.txt")` |
| `open(f).read()` | `Path(f).read_text(encoding="utf-8")` |
| `os.path.getsize(f)` | `Path(f).stat().st_size` |

**Hash** é uma conta que transforma qualquer arquivo numa sequência de tamanho fixo (no SHA-256, 64
caracteres). Mesma entrada → mesmo hash, sempre. Um byte diferente → hash completamente diferente
(efeito avalanche). Não dá para "voltar" do hash ao arquivo. Por isso o hash serve de **lacre**: anote-o
no momento da coleta; se depois bater, o arquivo é o mesmo.

**Cadeia de custódia** é a documentação da história de cada vestígio, da coleta ao descarte. Está nos
**arts. 158-A a 158-F do Código de Processo Penal**, incluídos pela **Lei 13.964/2019** (o "Pacote
Anticrime"). Para evidência digital, o manifesto com hash, data e responsável é a forma prática de
cumprir essa exigência.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/print_files_names.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/original/print_files_names.py) | Lista os `.txt` e depois todos os arquivos de `test_dir` |
| [`original/print_files_contents.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/original/print_files_contents.py) | Mostra o conteúdo de cada `.txt` com `readlines()` |
| [`original/check_files_contents.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/original/check_files_contents.py) | Imprime os arquivos que contêm o caractere `"2"` |
| [`original/check_re_files_contents.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/original/check_re_files_contents.py) | Imprime os arquivos que contêm um e-mail (regex do Dia 11) |
| [`original/test_dir/`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/original/test_dir) | Pasta de teste do autor (`file1.txt`, `file2.txt`, `file3.csv`) |

```bash
cd dia-14-sistema-de-arquivos/original     # os scripts originais esperam test_dir ao lado
python print_files_names.py
python check_re_files_contents.py
```

Nada para instalar. ⚠️ Rode **de dentro de `original/`**: o `os.chdir("test_dir")` procura a pasta
a partir de onde você está.

### `original/check_files_contents.py` — Procurar um texto dentro de vários arquivos

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Procurar um texto dentro de vários arquivos
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_14/check_files_contents.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────

import glob, os

os.chdir("test_dir")

# 💬 Percorre TODOS os arquivos da pasta (txt e csv).
for file in glob.glob("*"): 

    with open (file) as current_file:
        # 💬 read() traz o arquivo inteiro como uma string só.
        current_file_content=current_file.read()
        # 💬 "in" testa se o trecho "2" aparece em algum lugar do texto.
        if "2" in current_file_content:
             print(current_file_content)

### `original/check_re_files_contents.py` — Procurar e-mails (regex) em vários arquivos

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Procurar e-mails (regex) em vários arquivos
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_14/check_re_files_contents.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────

import glob, os, re 

# 💬 re.compile prepara a regex uma vez só (a mesma do Dia 11) para reusar no laço.
regexp = re.compile('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+')

os.chdir("test_dir")

for file in glob.glob("*"): 

    with open (file) as current_file:
        current_file_content=current_file.read()       
        # 💬 search procura o padrão em QUALQUER ponto do texto; devolve None se não achar.
        if regexp.search(current_file_content):
            # 💬 "Math" é um erro de digitação do autor para "Match" (correspondência). Mantido: é o original.
            print("Math in file: "+file)
            print(current_file_content)

### `original/print_files_contents.py` — Mostrar o conteúdo dos arquivos .txt

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Mostrar o conteúdo dos arquivos .txt
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_14/print_files_contents.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────

import glob, os

os.chdir("test_dir")

# 💬 Para cada .txt encontrado...
for file in glob.glob("*.txt"): 

    # 💬 ...abre o arquivo. Sem encoding=, o Python usa o padrão do sistema (no Windows pode ser cp1252).
    with open (file) as current_file:
        # 💬 readlines() devolve uma LISTA de linhas, cada uma terminando em \n.
        print(current_file.readlines())

### `original/print_files_names.py` — Listar nomes de arquivos de uma pasta

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Listar nomes de arquivos de uma pasta
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_14/print_files_names.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────

# 💬 glob acha arquivos por "curinga" (*.txt); os conversa com o sistema operacional.
import glob, os

# 💬 Entra na pasta test_dir. Por isso rode este script DE DENTRO de original/ (cd original).
os.chdir("test_dir")


print ("TXT files")
# 💬 "*.txt" = qualquer nome que termine em .txt.
for file in glob.glob("*.txt"): 
    print(file)

print ("ALL files")
# 💬 "*" = todos os arquivos (a ordem pode variar de sistema para sistema).
for file in glob.glob("*"): 
    print(file)

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão** e 100% offline. Nenhum script altera `caso/dados`.

| Arquivo | O que faz |
|---|---|
| [`br/01_pathlib_basico.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/br/01_pathlib_basico.py) | Anatomia de um caminho, `glob` × `rglob`, inventário com tamanho e data, e o original traduzido para `pathlib` |
| [`br/02_varre_evidencias.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/br/02_varre_evidencias.py) | Varre `caso/dados` atrás de CPFs, chaves PIX e e-mails; mostra quais valores ligam vários arquivos |
| [`br/03_hash_custodia.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/br/03_hash_custodia.py) | Manifesto SHA-256 (`gerar`), `verificar` e uma `demo` que adultera uma **cópia** e mostra a detecção |

```bash
cd dia-14-sistema-de-arquivos
python br/01_pathlib_basico.py
python br/02_varre_evidencias.py
python br/03_hash_custodia.py          # MODO = "demo" (troque para "gerar" ou "verificar")
```

**O que você deve ver no caso:** a chave `f38b2ffc-…` aparece em **6 arquivos** (vitrine, conversa,
planilha, dump e dois depoimentos). Os dois e-mails do vendedor ligam a vitrine à conversa. Na demo, a
troca de **uma letra** num depoimento é pega pelo hash.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_pathlib_basico.py` — pathlib: o jeito moderno (e que funciona no Windows) de andar por pastas.

In [ ]:
%cd -q {PASTA_NB}
"""
pathlib: o jeito moderno (e que funciona no Windows) de andar por pastas.

Carta 35 (rank A) — Jutsu da Sala de Evidências

O que você aprende:
  • a montar caminhos com Path e o operador / (sem se preocupar com \\ ou /);
  • glob (só na pasta) x rglob (na pasta e em TODAS as subpastas);
  • a ler tamanho e data de modificação com stat();
  • a "traduzir" os scripts originais (os.chdir + glob.glob) para pathlib.

Este script é 100% offline. Por padrão, passeia por caso/dados.

Como rodar (dentro da pasta dia-14-sistema-de-arquivos):
    python br/01_pathlib_basico.py
"""

import glob
import os
from datetime import datetime
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"   # "/" junta pedaços de caminho

# ── 1. Anatomia de um caminho ─────────────────────────────────────────────
exemplo = PASTA_CASO / "depoimentos" / "DEN-001.txt"
print("🧩 Anatomia de um caminho")
print(f"   caminho completo : {exemplo}")
print(f"   .name            : {exemplo.name}")        # nome com extensão
print(f"   .stem            : {exemplo.stem}")        # nome sem extensão
print(f"   .suffix          : {exemplo.suffix}")      # extensão
print(f"   .parent.name     : {exemplo.parent.name}") # pasta onde está
print(f"   .exists()        : {exemplo.exists()}")

# ── 2. glob x rglob ───────────────────────────────────────────────────────
print("\n📂 glob('*.txt') em caso/dados (só o primeiro nível):")
for arquivo in sorted(PASTA_CASO.glob("*.txt")):
    print(f"   {arquivo.name}")

print("\n📂 rglob('*.txt') em caso/dados (inclui subpastas):")
for arquivo in sorted(PASTA_CASO.rglob("*.txt")):
    # relative_to mostra o caminho a partir de caso/dados (mais fácil de ler)
    print(f"   {arquivo.relative_to(PASTA_CASO)}")


# ── 3. stat(): tamanho e data de modificação ──────────────────────────────
def tamanho_legivel(n):
    """1536 → '1.5 KB'."""
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:.0f} {unidade}" if unidade == "B" else f"{n:.1f} {unidade}"
        n /= 1024


print(f"\n📋 Inventário de caso/dados ({'arquivo':<34} {'tamanho':>9}  modificado em)")
total = 0
for arquivo in sorted(p for p in PASTA_CASO.rglob("*") if p.is_file()):
    info = arquivo.stat()
    total += info.st_size
    modificado = datetime.fromtimestamp(info.st_mtime)   # st_mtime vem em segundos desde 1970
    print(f"   {str(arquivo.relative_to(PASTA_CASO)):<42} {tamanho_legivel(info.st_size):>9}  {modificado:%d/%m/%Y %H:%M}")
print(f"   {'TOTAL':<42} {tamanho_legivel(total):>9}")

# ── 4. Original (os + glob) x pathlib, lado a lado ────────────────────────
# O original faz os.chdir("test_dir"): muda a pasta de trabalho do programa INTEIRO.
# Isso é perigoso em scripts maiores (todo caminho relativo passa a apontar para outro lugar).
# Com pathlib, não mudamos de pasta: só dizemos onde procurar.
test_dir = PASTA_DIA / "original" / "test_dir"
print("\n🔁 Original x pathlib (na pasta original/test_dir):")
jeito_original = sorted(os.path.basename(p) for p in glob.glob(os.path.join(test_dir, "*.txt")))
jeito_pathlib = sorted(p.name for p in test_dir.glob("*.txt"))
print(f"   glob.glob   → {jeito_original}")
print(f"   Path.glob   → {jeito_pathlib}")
print(f"   Iguais? {'✅ sim' if jeito_original == jeito_pathlib else '❌ não'}")

print("\n📖 Ler sem abrir/fechar na mão:")
for arquivo in sorted(test_dir.glob("*")):
    # read_text faz open + read + close de uma vez; encoding evita surpresa no Windows
    print(f"   {arquivo.name}: {arquivo.read_text(encoding='utf-8').strip()!r}")

### `br/02_varre_evidencias.py` — Varrer a sala de evidências: CPFs, chaves PIX e e-mails em todos os arquivos.

In [ ]:
%cd -q {PASTA_NB}
"""
Varrer a sala de evidências: CPFs, chaves PIX e e-mails em todos os arquivos.

Carta 35 (rank A) — Jutsu da Sala de Evidências (parte 2)

O que você aprende:
  • a juntar rglob (Dia 14) com regex (Dia 11) para varrer uma pasta inteira;
  • a pular arquivos que não são texto sem o script quebrar;
  • a montar um relatório "arquivo × tipo de dado" e um índice reverso
    "valor → arquivos onde aparece" (ótimo para achar ligações);
  • a gravar o relatório em CSV para abrir no Excel.

Este script é 100% offline: varre caso/dados (fictício).

Como rodar (dentro da pasta dia-14-sistema-de-arquivos):
    python br/02_varre_evidencias.py
"""

import csv
import re
from collections import defaultdict
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
# Caminho relativo à pasta do dia (ou absoluto, se preferir).
PASTA_ALVO = "../caso/dados"  #@param {type:"string"}
EXTENSOES = ".txt,.csv,.html,.json,.sql"  #@param {type:"string"}

PADROES = {
    "CPF": re.compile(r"(?<!\d)\d{3}\.\d{3}\.\d{3}-\d{2}(?!\d)"),
    "Chave PIX": re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b", re.I),
    "E-mail": re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"),
}

pasta = (PASTA_DIA / PASTA_ALVO).resolve()   # "/" com caminho absoluto usa só o absoluto
extensoes = {e.strip().lower() for e in EXTENSOES.split(",")}

relatorio = []                       # uma linha por arquivo
onde_aparece = defaultdict(set)      # (tipo, valor) → arquivos
pulados = []

# ── 1. Percorrer todos os arquivos, em todas as subpastas ─────────────────
for arquivo in sorted(pasta.rglob("*")):
    if not arquivo.is_file() or arquivo.suffix.lower() not in extensoes:
        continue
    if arquivo.name == "ficha_do_caso.json":
        continue      # é o gabarito do caso: não conta como evidência
    try:
        texto = arquivo.read_text(encoding="utf-8")
    except (UnicodeDecodeError, OSError) as erro:
        pulados.append(f"{arquivo.name} ({type(erro).__name__})")
        continue

    nome = str(arquivo.relative_to(pasta))
    linha = {"arquivo": nome}
    for tipo, padrao in PADROES.items():
        achados = set(padrao.findall(texto))          # set: cada valor uma vez por arquivo
        linha[tipo] = len(achados)
        for valor in achados:
            onde_aparece[(tipo, valor.lower())].add(nome)
    relatorio.append(linha)

# ── 2. Tabela arquivo × tipo ──────────────────────────────────────────────
print(f"🗂️  Varredura de {pasta.name}/  ({len(relatorio)} arquivos lidos)\n")
print(f"   {'arquivo':<34} {'CPF':>5} {'PIX':>5} {'E-mail':>7}")
for l in relatorio:
    print(f"   {l['arquivo']:<34} {l['CPF']:>5} {l['Chave PIX']:>5} {l['E-mail']:>7}")
if pulados:
    print(f"\n   ⚠️ Pulados (não são texto UTF-8): {', '.join(pulados)}")

# ── 3. Índice reverso: o que aparece em MAIS arquivos ─────────────────────
print("\n🔗 Valores que ligam vários arquivos (PIX e e-mail em 2+ arquivos):")
for (tipo, valor), arquivos in sorted(onde_aparece.items(), key=lambda par: -len(par[1])):
    if tipo != "CPF" and len(arquivos) >= 2:
        print(f"   {tipo:<9} {valor}  → {len(arquivos)} arquivos")
        print(f"             {', '.join(sorted(arquivos))}")

cpfs = [v for (t, v) in onde_aparece if t == "CPF"]
print(f"\n🔒 {len(cpfs)} CPFs distintos encontrados (dados de vítimas: não listados na tela).")

# ── 4. Salvar ─────────────────────────────────────────────────────────────
saidas = PASTA_DIA / "saidas"
saidas.mkdir(exist_ok=True)
with open(saidas / "varredura_evidencias.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=["arquivo", *PADROES], delimiter=";")
    escritor.writeheader()
    escritor.writerows(relatorio)
print("💾 saidas/varredura_evidencias.csv gravado.")

### `br/03_hash_custodia.py` — Cadeia de custódia na prática: manifesto SHA-256 das evidências e verificação.

In [ ]:
%cd -q {PASTA_NB}
"""
Cadeia de custódia na prática: manifesto SHA-256 das evidências e verificação.

Carta 35 (rank A) — Jutsu da Sala de Evidências (parte 3)

O que você aprende:
  • o que é um hash (uma "impressão digital" de um arquivo) e por que SHA-256;
  • a calcular o hash lendo o arquivo em BLOCOS (funciona até com arquivos de vários GB);
  • a gerar um manifesto (CSV) com caminho, tamanho, data de modificação e hash;
  • a VERIFICAR depois: o script avisa se algum arquivo foi alterado, sumiu ou apareceu.

Dois modos (parâmetro MODO):
  "gerar"     → cria saidas/manifesto_custodia.csv
  "verificar" → compara a pasta com um manifesto já existente
  "demo"      → gera, simula uma adulteração numa CÓPIA e mostra a verificação pegando

Este script é 100% offline e NUNCA altera caso/dados (a demo mexe só numa cópia em saidas/).

Como rodar (dentro da pasta dia-14-sistema-de-arquivos):
    python br/03_hash_custodia.py
"""

import csv
import hashlib
import shutil
from datetime import datetime, timedelta, timezone
from pathlib import Path

MODO = "demo"  #@param ["gerar", "verificar", "demo"]
PASTA_ALVO = "../caso/dados"  #@param {type:"string"}
TAMANHO_BLOCO = 65536  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
SAIDAS = PASTA_DIA / "saidas"
MANIFESTO = SAIDAS / "manifesto_custodia.csv"

# Fuso de Brasília (com plano B para Windows sem tzdata).
try:
    from zoneinfo import ZoneInfo
    FUSO_BR = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO_BR = timezone(timedelta(hours=-3))


def sha256_do_arquivo(caminho):
    """Lê o arquivo aos pedaços e vai alimentando o hash. Memória usada: só 1 bloco."""
    h = hashlib.sha256()
    with open(caminho, "rb") as arquivo:          # "rb" = leitura BINÁRIA: o hash é dos bytes
        while True:
            bloco = arquivo.read(TAMANHO_BLOCO)
            if not bloco:                          # b"" = chegou ao fim
                break
            h.update(bloco)
    return h.hexdigest()                           # 64 caracteres hexadecimais


def gerar_manifesto(pasta, destino):
    """Percorre a pasta e grava uma linha por arquivo no CSV de manifesto."""
    linhas = []
    for arquivo in sorted(p for p in pasta.rglob("*") if p.is_file()):
        info = arquivo.stat()
        linhas.append({
            # as_posix() grava com "/" mesmo no Windows: o manifesto vale em qualquer sistema
            "arquivo": arquivo.relative_to(pasta).as_posix(),
            "bytes": info.st_size,
            "modificado_em": datetime.fromtimestamp(info.st_mtime, FUSO_BR).isoformat(timespec="seconds"),
            "sha256": sha256_do_arquivo(arquivo),
        })
    destino.parent.mkdir(parents=True, exist_ok=True)
    with open(destino, "w", newline="", encoding="utf-8") as f:
        f.write(f"# Manifesto de {pasta.name}/ gerado em {datetime.now(FUSO_BR):%d/%m/%Y %H:%M:%S} (Brasília)\n")
        escritor = csv.DictWriter(f, fieldnames=["arquivo", "bytes", "modificado_em", "sha256"], delimiter=";")
        escritor.writeheader()
        escritor.writerows(linhas)
    return linhas


def ler_manifesto(caminho):
    with open(caminho, encoding="utf-8") as f:
        linhas_uteis = [l for l in f if not l.startswith("#")]   # pula o comentário do topo
    return {l["arquivo"]: l for l in csv.DictReader(linhas_uteis, delimiter=";")}


def verificar(pasta, manifesto):
    """Compara o estado atual da pasta com o manifesto. Devolve a quantidade de problemas."""
    esperado = ler_manifesto(manifesto)
    atual = {p.relative_to(pasta).as_posix(): p for p in pasta.rglob("*") if p.is_file()}
    problemas = 0
    for nome in sorted(set(esperado) | set(atual)):
        if nome not in atual:
            print(f"   ❌ SUMIU      {nome}")
            problemas += 1
        elif nome not in esperado:
            print(f"   ➕ NOVO       {nome}  (não estava no manifesto)")
            problemas += 1
        elif sha256_do_arquivo(atual[nome]) != esperado[nome]["sha256"]:
            print(f"   🚩 ALTERADO   {nome}  (hash diferente)")
            problemas += 1
        else:
            print(f"   ✅ íntegro    {nome}")
    return problemas


pasta = (PASTA_DIA / PASTA_ALVO).resolve()

# ── Demonstração rápida: um caractere muda o hash inteiro ─────────────────
print("🔬 Efeito avalanche:")
for frase in ["Chave PIX: f38b2ffc", "Chave PIX: f38b2ffd"]:
    print(f"   {frase!r:<24} → {hashlib.sha256(frase.encode()).hexdigest()[:32]}…")

if MODO == "gerar":
    linhas = gerar_manifesto(pasta, MANIFESTO)
    print(f"\n🧾 {len(linhas)} arquivos no manifesto → {MANIFESTO.relative_to(PASTA_DIA)}")

elif MODO == "verificar":
    if not MANIFESTO.exists():
        raise SystemExit("Não há manifesto ainda. Rode antes com MODO = 'gerar'.")
    print(f"\n🔍 Verificando {pasta.name}/ contra o manifesto:")
    n = verificar(pasta, MANIFESTO)
    print(f"\n{'✅ Tudo íntegro.' if n == 0 else f'🚩 {n} problema(s): registre e investigue antes de usar a evidência.'}")

else:   # demo
    # 1) Copiamos as evidências para saidas/ (o original fica intocado)
    copia = SAIDAS / "copia_de_trabalho"
    if copia.exists():
        shutil.rmtree(copia)
    shutil.copytree(pasta, copia)       # copytree preserva as datas de modificação
    linhas = gerar_manifesto(copia, MANIFESTO)
    print(f"\n🧾 Manifesto gerado com {len(linhas)} arquivos. Amostra:")
    for l in linhas[:4]:
        print(f"   {l['sha256'][:16]}…  {l['bytes']:>6} B  {l['arquivo']}")

    # 2) "Alguém" mexe na cópia: troca a chave PIX de um depoimento e cria um arquivo novo
    alvo = copia / "depoimentos" / "DEN-001.txt"
    texto = alvo.read_text(encoding="utf-8")
    alvo.write_text(texto.replace("f38b2ffc", "f38b2ffd"), encoding="utf-8")
    (copia / "anotacao_solta.txt").write_text("rascunho\n", encoding="utf-8")
    print("\n😈 Simulação: 1 letra trocada em DEN-001.txt e 1 arquivo novo criado.")

    # 3) A verificação pega as duas coisas
    print("\n🔍 Verificação:")
    n = verificar(copia, MANIFESTO)
    print(f"\n🚩 {n} problema(s) detectado(s). Sem o manifesto, ninguém perceberia a troca de 1 letra.")

## ⚖️ Ética, LGPD e OPSEC

- **Nunca trabalhe no original.** Faça a cópia de trabalho, calcule o hash do original **antes**, e
  analise só a cópia. A demo do script 03 segue essa regra.
- **Hash no momento da coleta.** Um hash calculado uma semana depois prova só que o arquivo não mudou
  *desde então*. Registre também quem coletou, quando, de onde e com qual ferramenta (art. 158-B do
  CPP: as etapas vão do reconhecimento ao descarte).
- **SHA-256, não MD5.** MD5 e SHA-1 têm colisões conhecidas (dois arquivos diferentes com o mesmo
  hash). Se um laudo antigo traz MD5, calcule também o SHA-256.
- **Abrir arquivo altera metadados.** Abrir um documento num editor pode mudar a data de acesso ou até
  regravar o arquivo. Ler com Python em modo `"rb"` não altera o conteúdo, mas em perícia de verdade
  usa-se bloqueador de escrita e imagem forense.
- **Varredura acha dados de terceiros.** O script 02 encontra 38 CPFs, quase todos de vítimas. Ele
  mostra só a contagem na tela: é minimização (LGPD) na prática.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Qual a diferença entre Path('casos').glob('*.txt') e Path('casos').rglob('*.txt')?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Nenhuma", "B) rglob também procura dentro de todas as subpastas", "C) rglob só acha arquivos ocultos", "D) glob é mais lento"]
correta = "B) rglob também procura dentro de todas as subpastas"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "glob olha só o primeiro nível; rglob desce recursivamente em todas as subpastas.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Você trocou uma única letra num depoimento. O que acontece com o SHA-256 dele?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Muda só um caractere do hash", "B) Não muda, porque o tamanho é o mesmo", "C) Muda completamente (efeito avalanche)", "D) Fica zerado"]
correta = "C) Muda completamente (efeito avalanche)"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Qualquer alteração de um byte gera um hash totalmente diferente; é isso que permite detectar adulteração.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Onde a cadeia de custódia está prevista na lei brasileira?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Na LGPD, art. 7º", "B) No Marco Civil da Internet", "C) Nos arts. 158-A a 158-F do CPP, incluídos pela Lei 13.964/2019", "D) No Código Civil"]
correta = "C) Nos arts. 158-A a 158-F do CPP, incluídos pela Lei 13.964/2019"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O Pacote Anticrime (Lei 13.964/2019) incluiu os arts. 158-A a 158-F no Código de Processo Penal.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 14: Triagem do pendrive apreendido

**Carta 56 · Rank S**

## A missão

Chega ao NFA a cópia de um pendrive (fictício) encontrado com um dos laranjas:
[`dados/pendrive_apreendido/`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-14-sistema-de-arquivos/dados/pendrive_apreendido). Antes de abrir qualquer coisa, a Iara quer
uma triagem automática que responda três perguntas:

1. **Algum arquivo mente sobre o que é?** (um `.txt` que na verdade é imagem, um `.pdf` que é ZIP…)
2. **Há duplicatas?** (mesmo conteúdo com nomes diferentes)
3. **O que já temos?** (arquivos idênticos a evidências de `caso/dados`)

Escreva um script que, para cada arquivo do pendrive (com subpastas):

- calcule o **SHA-256** em blocos;
- leia os **primeiros bytes** e descubra o tipo real pelo "número mágico" (PNG começa com
  `\x89PNG`, PDF com `%PDF-`, ZIP com `PK\x03\x04`…);
- compare o tipo real com a extensão e marque 🎭 se não baterem;
- marque ♊ as duplicatas e 📁 o que já existe em `caso/dados`;
- grave `saidas/triagem_pendrive.csv` (`;`, `utf-8-sig`).

## Critérios de rank S

- [ ] Só biblioteca padrão (`hashlib`, `pathlib`, `csv`, `collections`)
- [ ] Lê em modo binário (`"rb"`) e em blocos
- [ ] Não altera nenhum arquivo do pendrive
- [ ] Acha as 2 extensões falsas, a duplicata e os 2 arquivos já conhecidos
- [ ] Diz quantos arquivos são **realmente novos** para analisar

## Dica

Monte um dicionário `hash → caminho` com todos os arquivos de `caso/dados` **antes** de olhar o
pendrive. Depois, "já conhecido?" vira só `hash in conhecidos`.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 14 — Triagem do pendrive apreendido.

Varre dados/pendrive_apreendido/ (fictício) e, para cada arquivo:
  • calcula o SHA-256 em blocos;
  • descobre o tipo REAL pelos primeiros bytes ("número mágico") e compara com a extensão;
  • marca duplicatas (mesmo hash, nomes diferentes);
  • marca o que já é conhecido (hash igual a algum arquivo de caso/dados).
Grava saidas/triagem_pendrive.csv. 100% offline.
"""

import csv
import hashlib
from collections import defaultdict
from pathlib import Path

PASTA_PENDRIVE = "dados/pendrive_apreendido"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

# Assinaturas de início de arquivo ("magic numbers") mais comuns.
MAGICOS = [
    (b"\x89PNG\r\n\x1a\n", "png"),
    (b"\xff\xd8\xff", "jpg"),
    (b"%PDF-", "pdf"),
    (b"PK\x03\x04", "zip"),            # também é o começo de .docx, .xlsx, .apk…
    (b"GIF8", "gif"),
    (b"SQLite format 3\x00", "sqlite"),
]
# Extensões que aceitamos como "compatíveis" com cada tipo real.
COMPATIVEIS = {
    "png": {".png"}, "jpg": {".jpg", ".jpeg"}, "pdf": {".pdf"}, "gif": {".gif"},
    "zip": {".zip", ".docx", ".xlsx", ".pptx", ".odt", ".apk", ".jar"},
    "sqlite": {".db", ".sqlite", ".sqlite3"},
    "texto": {".txt", ".csv", ".html", ".htm", ".json", ".sql", ".md", ".xml"},
}


def sha256(caminho, bloco=65536):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for pedaco in iter(lambda: f.read(bloco), b""):   # lê até devolver b"" (fim)
            h.update(pedaco)
    return h.hexdigest()


def tipo_real(caminho):
    with open(caminho, "rb") as f:
        inicio = f.read(16)
    for assinatura, tipo in MAGICOS:
        if inicio.startswith(assinatura):
            return tipo
    try:
        inicio.decode("utf-8")       # não é binário conhecido; se decodifica, tratamos como texto
        return "texto"
    except UnicodeDecodeError:
        return "desconhecido"


# ── 1. Hashes já conhecidos (as evidências que o NFA já tem) ──────────────
conhecidos = {sha256(p): p.relative_to(PASTA_CASO).as_posix() for p in PASTA_CASO.rglob("*") if p.is_file()}

# ── 2. Varredura do pendrive ──────────────────────────────────────────────
pendrive = PASTA_DIA / PASTA_PENDRIVE
linhas, por_hash = [], defaultdict(list)
for arquivo in sorted(p for p in pendrive.rglob("*") if p.is_file()):
    try:
        h, tipo = sha256(arquivo), tipo_real(arquivo)
    except OSError as erro:
        print(f"⚠️  Não consegui ler {arquivo.name}: {erro}")
        continue
    nome = arquivo.relative_to(pendrive).as_posix()
    extensao_ok = arquivo.suffix.lower() in COMPATIVEIS.get(tipo, set())
    linhas.append({"arquivo": nome, "bytes": arquivo.stat().st_size, "tipo_real": tipo,
                   "extensao_ok": "sim" if extensao_ok else "NÃO", "sha256": h,
                   "ja_conhecido": conhecidos.get(h, ""), "duplicata_de": ""})
    por_hash[h].append(nome)

# ── 3. Marcar duplicatas (o primeiro nome em ordem alfabética é o "original") ─
for linha in linhas:
    irmaos = por_hash[linha["sha256"]]
    if len(irmaos) > 1 and linha["arquivo"] != irmaos[0]:
        linha["duplicata_de"] = irmaos[0]

# ── 4. Relatório ──────────────────────────────────────────────────────────
print(f"🔌 Triagem de {PASTA_PENDRIVE}/ ({len(linhas)} arquivos)\n")
for l in linhas:
    alertas = []
    if l["extensao_ok"] == "NÃO":
        alertas.append(f"🎭 extensão mente: é {l['tipo_real'].upper()}")
    if l["duplicata_de"]:
        alertas.append(f"♊ cópia de {l['duplicata_de']}")
    if l["ja_conhecido"]:
        alertas.append(f"📁 já temos: {l['ja_conhecido']}")
    print(f"   {l['sha256'][:12]}…  {l['arquivo']:<28} {' · '.join(alertas) or '🆕 novo'}")

novos = [l for l in linhas if not l["ja_conhecido"] and not l["duplicata_de"]]
print(f"\n📊 {len(linhas)} arquivos · {len(novos)} realmente novos para analisar · "
      f"{sum(l['extensao_ok'] == 'NÃO' for l in linhas)} com extensão falsa")

saidas = PASTA_DIA / "saidas"
saidas.mkdir(exist_ok=True)
with open(saidas / "triagem_pendrive.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=list(linhas[0]), delimiter=";")
    escritor.writeheader()
    escritor.writerows(linhas)
print("💾 saidas/triagem_pendrive.csv gravado.")

## 🔎 Inovação OSINT do dia

**Hash "difuso" (fuzzy hashing) com `ppdeep`.** O SHA-256 diz só "igual" ou "diferente": mudou uma
vírgula, o hash não tem nada a ver. Já o *fuzzy hashing* (algoritmo do **ssdeep**) gera hashes que
ficam **parecidos** quando os arquivos são parecidos, e dá uma nota de semelhança de 0 a 100. Serve
para achar a mesma vitrine falsa com o preço trocado, ou versões de um mesmo documento. A biblioteca
`ppdeep` implementa o ssdeep em Python puro:

```python
# pip install ppdeep
import ppdeep
a = ppdeep.hash(open("vitrine_v1.html", "rb").read())
b = ppdeep.hash(open("vitrine_v2.html", "rb").read())
print(ppdeep.compare(a, b))   # 0 a 100
```

Use o SHA-256 para **integridade** (prova) e o fuzzy hash para **semelhança** (pista).

## 📚 Para ir além

- `pathlib` (documentação oficial): <https://docs.python.org/pt-br/3/library/pathlib.html>
- `hashlib`: <https://docs.python.org/pt-br/3/library/hashlib.html>
- Código de Processo Penal, arts. 158-A a 158-F: <https://www.planalto.gov.br/ccivil_03/decreto-lei/del3689.htm>
- Lei 13.964/2019: <https://www.planalto.gov.br/ccivil_03/_ato2019-2022/2019/lei/l13964.htm>
- Lista de assinaturas de arquivo ("magic numbers"): <https://en.wikipedia.org/wiki/List_of_file_signatures>
- ppdeep: <https://github.com/elceef/ppdeep>

## ✅ Checklist

- [ ] Rodei os quatro scripts originais de dentro de `original/`
- [ ] Usei `rglob` para listar todos os arquivos do caso, com subpastas
- [ ] Descobri em quantos arquivos aparece cada chave PIX
- [ ] Gerei o manifesto SHA-256 e vi a demo pegar a adulteração
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 14](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)